In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata
from scipy.optimize import minimize
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("--- [Monolith v6.1] Full Pipeline with Advanced Factors, Early Stopping & Nelder-Mead ---")

# ==========================================
# 1. ЗАГРУЗКА И ОЧИСТКА ДАННЫХ
# ==========================================
print("[1/5] Loading data...")
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test .csv')


TARGET = 'Will_Buy_EV'
ID_COL = 'id'

train = train.dropna(subset=[TARGET])
if train[TARGET].dtype == 'object':
    train[TARGET] = train[TARGET].replace({'Yes': 1, 'No': 0})
train[TARGET] = train[TARGET].astype(np.int32)

base_features = [c for c in train.columns if c not in [TARGET, ID_COL]]

# Факторизация текстовых колонок под GPU
cat_cols = train[base_features].select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    train[col], indexer = pd.factorize(train[col])
    test[col] = test[col].map(lambda x: indexer.get_loc(x) if x in indexer else -1)

# ==========================================
# 2. ПРОЕКТИРОВАНИЕ ПСИХОЛОГИЧЕСКИХ И ЭКОНОМИЧЕСКИХ ФАКТОРОВ
# ==========================================
print("[2/5] Synthesizing advanced features...")
def build_psychological_factors(df):
    df_out = df.copy()
    eps = 1e-5

    # Индекс Консервативного Автовладельца (Высокий возраст * много машин)
    df_out['loyal_conservative_index'] = df_out['Age'] * df_out['Number_of_Cars_Owned']

    # Стены консерватизма против субсидий
    df_out['conservative_vs_subsidy'] = df_out['loyal_conservative_index'] * (1.0 / (df_out['Subsidy_Available'] + 0.5))

    # Индекс Раннего Последователя (Молодой Новатор: доход / возраст)
    df_out['early_adopter_index'] = df_out['Annual_Income_USD'] / (df_out['Age'] + eps)
    df_out['young_eco_trendsetter'] = df_out['early_adopter_index'] * (df_out['Environmental_Concern_Level'] + 1)

    # Экономическая инерция (Маленький ежедневный пробег при высоком доходе)
    df_out['low_commute_high_income_inertia'] = df_out['Annual_Income_USD'] / (df_out['Daily_Commute_km'] + 1.0)

    # Разрешение парадокса Симпсона (Зарядки у дома + Домашняя розетка)
    df_out['home_charge_vs_stations'] = df_out['Home_Charging_Possible'].astype(str) + "_" + df_out['Charging_Stations_Near_Home'].astype(str)
    df_out['home_charge_vs_stations'], _ = pd.factorize(df_out['home_charge_vs_stations'])

    # Инфраструктурный эко-скор
    total_charging = df_out['Charging_Stations_Near_Home'] + df_out['Charging_Stations_Near_Work']
    df_out['eco_charging_score'] = total_charging * (df_out['Environmental_Concern_Level'] + 1)

    # Геометрические отношения
    df_out['Age_ratio_Annual_Income_USD'] = df_out['Age'] / (df_out['Annual_Income_USD'] + eps)
    df_out['Age_ratio_Daily_Commute_km'] = df_out['Age'] / (df_out['Daily_Commute_km'] + eps)
    df_out['Annual_Income_USD_ratio_Daily_Commute_km'] = df_out['Annual_Income_USD'] / (df_out['Daily_Commute_km'] + eps)
    df_out['commute_anxiety_stress'] = df_out['Daily_Commute_km'] * df_out['Range_Anxiety_Level']

    return df_out

train_adv = build_psychological_factors(train)
test_adv = build_psychological_factors(test)

all_features = [c for c in train_adv.columns if c not in [TARGET, ID_COL]]

# Подготовка float32 матриц для видеокарты
X_full = train_adv[all_features].astype(np.float32)
y_full = train_adv[TARGET].astype(np.int32)
X_test_mod = test_adv[all_features].astype(np.float32)

# Интеграция мета-сигналов шума из нашего аудитора данных
X_full['gen_error_signal'] = dirty_rows_report['generation_error'].values
X_full['is_bluff_signal'] = dirty_rows_report['is_entropy_bluff'].astype(np.float32).values

X_test_mod['gen_error_signal'] = dirty_rows_report['generation_error'].median()
X_test_mod['is_bluff_signal'] = 0.0

print(f"-> Feature Matrices assembled. Total columns: {X_full.shape[1]}")

# ==========================================
# 3. НАСТРОЙКА И ОБУЧЕНИЕ С РАННЕЙ ОСТАНОВКОЙ (GPU)
# ==========================================
print("[3/5] Starting calibrated training with Early Stopping...")
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Зажимаем регуляризацию (max_depth=6) и снижаем learning_rate до 0.02
lgb_params = {
    'n_estimators': 2500, 'learning_rate': 0.02, 'objective': 'binary',
    'metric': 'auc', 'random_state': 42, 'verbose': -1, 'device': 'gpu',
    'colsample_bytree': 0.65, 'subsample': 0.8, 'max_depth': 6, 'num_leaves': 31
}

xgb_params = {
    'n_estimators': 2500, 'learning_rate': 0.02, 'objective': 'binary:logistic',
    'eval_metric': 'auc', 'random_state': 42, 'tree_method': 'hist', 'device': 'cuda',
    'colsample_bytree': 0.65, 'subsample': 0.8, 'max_depth': 6, 'min_child_weight': 5
}

cat_params = {
    'iterations': 2500, 'learning_rate': 0.02, 'loss_function': 'Logloss',
    'eval_metric': 'AUC', 'random_state': 42, 'verbose': False, 'task_type': 'GPU',
    'l2_leaf_reg': 7, 'depth': 6
}

oof_lgb, test_lgb = np.zeros(len(X_full)), np.zeros(len(X_test_mod))
oof_xgb, test_xgb = np.zeros(len(X_full)), np.zeros(len(X_test_mod))
oof_cat, test_cat = np.zeros(len(X_full)), np.zeros(len(X_test_mod))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    X_train, y_train = X_full.iloc[train_idx], y_full.iloc[train_idx]
    X_val, y_val = X_full.iloc[val_idx], y_full.iloc[val_idx]

    # 1. LightGBM (контролируем переобучение через ограничение итераций)
    model_lgb = LGBMClassifier(**lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_lgb += model_lgb.predict_proba(X_test_mod)[:, 1] / N_SPLITS

    # 2. XGBoost с жестким Early Stopping
    model_xgb = XGBClassifier(**xgb_params, early_stopping_rounds=50)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_xgb += model_xgb.predict_proba(X_test_mod)[:, 1] / N_SPLITS

    # 3. CatBoost с жестким Early Stopping
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, use_best_model=True)
    oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_cat += model_cat.predict_proba(X_test_mod)[:, 1] / N_SPLITS

    print(f" -> Fold {fold+1}/{N_SPLITS} successfully calibrated.")

# ==========================================
# 4. МАТЕМАТИЧЕСКАЯ ОПТИМИЗАЦИЯ ВЕСОВ (Nelder-Mead Rank Fusion)
# ==========================================
print("\n[4/5] Optimizing ensemble weights via Nelder-Mead...")

def objective_func(weights):
    w = weights / np.sum(weights)
    blend_oof = (
        w[0] * (rankdata(oof_lgb) / len(oof_lgb)) +
        w[1] * (rankdata(oof_xgb) / len(oof_xgb)) +
        w[2] * (rankdata(oof_cat) / len(oof_cat))
    )
    return -roc_auc_score(y_full, blend_oof)

initial_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]

res = minimize(objective_func, initial_weights, method='Nelder-Mead', bounds=bounds)
best_weights = res.x / np.sum(res.x)
best_auc = -res.fun

print("="*60)
print("🎯 BLENDING CALIBRATION COMPLETE 🎯")
print("="*60)
print(f"Optimal weights: LightGBM={best_weights[0]:.4f} | XGBoost={best_weights[1]:.4f} | CatBoost={best_weights[2]:.4f}")
print(f"🚀 Optimized OOF AUC: {best_auc:.5f}\n")

# ==========================================
# 5. СОРЕВНОВАТЕЛЬНЫЙ ВЗВЕШЕННЫЙ САБМИТ
# ==========================================
print("[5/5] Generating final submission...")
final_test_preds = (
    best_weights[0] * (rankdata(test_lgb) / len(test_lgb)) +
    best_weights[1] * (rankdata(test_xgb) / len(test_xgb)) +
    best_weights[2] * (rankdata(test_cat) / len(test_cat))
)

submission = pd.DataFrame({
    'id': test['id'],
    'Will_Buy_EV': final_test_preds
})
submission.to_csv('submission_monolith_calibrated.csv', index=False)
print("Файл 'submission_monolith_calibrated.csv' готов! Загружай на лидерборд.")

In [ ]:
import numpy as np
import pandas as pd

# 1. Безопасно собираем среднее предсказание Триады из памяти ноутбука
try:
    mean_oof_prob = (oof_lgb + oof_xgb + oof_cat) / 3
    print("--- [Engine] Out-of-fold предсказания успешно загружены ---")
except NameError:
    print("❌ Ошибка: Модели еще не обучены в этой сессии или переменные oof_lgb/xgb/cat удалены!")

# 2. Монолитная функция аудита косяков генератора
def audit_synthetic_failures_secure(X_data, y_true, oof_preds):
    report = X_data.copy()
    report['true_target'] = y_true
    report['model_prob'] = oof_preds

    # Абсолютная ошибка генерации (где модель и таргет жестко противоречат друг другу)
    report['generation_error'] = np.abs(report['true_target'] - report['model_prob'])

    # Зона Блефа (Энтропийный тупик вокруг 0.5)
    report['is_entropy_bluff'] = (report['model_prob'] >= 0.4) & (report['model_prob'] <= 0.6)

    # Сортируем по максимальной ошибке
    top_failures = report.sort_values(by='generation_error', ascending=False)

    print("="*60)
    print("🚨 [DATA QUALITY AUDITOR] ОТЧЕТ О КОСЯКАХ СИНТЕТИКИ 🚨")
    print("="*60)
    total_rows = len(report)
    bluff_count = report['is_entropy_bluff'].sum()
    critical_count = (report['generation_error'] > 0.8).sum()

    print(f"Всего строк в тупике энтропии (модель бессильна): {bluff_count} ({bluff_count/total_rows*100:.2f}%)")
    print(f"Строк с критическим искажением логики (Ошибка > 0.8): {critical_count} ({critical_count/total_rows*100:.2f}%)")
    print("-"*60)

    # Автоматический вывод топ-5 самых бредовых строк, сгенерированных CTGAN
    print("🔍 ТОП-5 строк с максимальным системным шумом:")
    display_cols = [c for c in report.columns if c not in ['is_entropy_bluff']]
    # Выведем первые несколько колонок для наглядности (чтобы не перегружать экран)
    print(top_failures[display_cols].head(5).to_string())

    return top_failures

# Запуск аудитора
dirty_rows_report = audit_synthetic_failures_secure(X, y, mean_oof_prob)


In [ ]:
import pandas as pd
import numpy as np

print("--- [Engine Intelligence] Извлечение ключевых факторов ---")

# Извлекаем важность признаков из обученных на последнем фолде моделей
# (Если ячейка обучения уже отработала, эти модели доступны в памяти)
try:
    lgb_imp = model_lgb.feature_importances_
    xgb_imp = list(model_xgb.get_booster().get_score(importance_type='weight').values())

    # Так как XGBoost возвращает словарь только для использованных признаков,
    # выровняем его по списку all_features
    xgb_features_dict = model_xgb.get_booster().get_score(importance_type='weight')
    xgb_imp = [xgb_features_dict.get(f, 0) for f in X_full.columns]

    cat_imp = model_cat.get_feature_importance()

    # Нормализуем важности (переводим в проценты от 0 до 100), чтобы их можно было усреднить
    lgb_imp_norm = 100 * (lgb_imp / np.sum(lgb_imp))
    xgb_imp_norm = 100 * (xgb_imp / np.sum(xgb_imp))
    cat_imp_norm = 100 * (cat_imp / np.sum(cat_imp))

    # Считаем среднюю синергетическую важность Триады
    triad_importance = (lgb_imp_norm + xgb_imp_norm + cat_imp_norm) / 3

    # Создаем итоговый датафрейм факторов
    importance_df = pd.DataFrame({
        'Фактор (Feature)': X_full.columns,
        'Важность для Триады (%)': triad_importance
    }).sort_values(by='Важность для Триады (%)', ascending=False).reset_index(drop=True)

    print("\n" + "="*50)
    print("📊 РЕЙТИНГ КЛЮЧЕВЫХ ФАКТОРОВ (TOP FEATURES) 📊")
    print("="*50)
    print(importance_df.to_string(index=True))

except NameError as e:
    print(f"❌ Ошибка: Убедитесь, что модели Триады успешно обучены в текущей сессии! Код ошибки: {e}")